# Week 16 – Research Assistant Agent with Tool Calls 16.3 Graded Mini Project
### Submitted by: Harley Dangan
### Date: 26 November 2026
### Submitted to: The Asian Institute of Management



## Load Environment Variables

This Jupyter notebook assumes that you have the environment variables configured. See .venv.example for reference.
***NOTE: langchain_ollama is used in this notebook. This notebook will only work with Ollama. This gives me unlimited tokens.***

In [ ]:
import os

base_url = os.environ.get("BASE_URL")
llm= os.environ.get("LLM")
api_key = os.environ.get("OPENAI_API_KEY") # Actually, any string will work here since I used Ollama!
tavily_api_key = os.environ.get("TAVILY_API_KEY")

if not base_url or not llm or not api_key or not tavily_api_key:
    raise ValueError("Missing required environment variables: BASE_URL, LLM, API_KEY, or TAVILY_API_KEY")
print(f"Base URL: {base_url}")
print(f"LLM: {llm}")
print(f"API Key: {api_key}")
print(f"Tavily API Key: {tavily_api_key}")

Base URL: http://localhost:11434/v1
LLM: gemma-4-26B-A4B-it-UD-IQ4_XS
API Key: ollama
Tavily API Key: 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


## Create LangGraph State Class

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# State is one shared object every node reads from and writes to. It is a TypedDict that defines the structure of the state object.
class AgentState(TypedDict):
    question: str
    answer: str
    messages: Annotated[list, add_messages]

## Initialize OpenAI-Compatible LLM and Tools

In [ ]:
import json
import os
from openai import OpenAI
from tavily import TavilyClient

# Initializes the client
client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

# Define the Travily Search Tool Schema

tools = [
    {
        "type": "function",
        "function": {
            "name": "tavily_search",
            "description": "Search the live web for current facts, news, and technical data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The specific search query optimized for a search engine. Do not include punctuation or conversational filler."
                    }
                },
                "required": ["query"]
            }
        }
    }
]

# Defining the system prompt for the model
system_prompt = """You are an expert Research Assistant Agent. Your core objective is to deliver comprehensive, highly accurate, and fact-verified research on any given topic. You have access to tools for searching the web and retrieving full webpage content.

### OPERATIONAL MANDATES
1. FACTUAL GROUNDING: Rely exclusively on verifiable information retrieved from your tools. If a fact cannot be sourced from your tool outputs, explicitly state that it is unavailable. Never assume, extrapolate, or hallucinate details.
2. LATENT SEARCH QUERIES: When an initial tool result is ambiguous, outdated, or incomplete, proactively formulate distinct follow-up queries or browse primary source URLs to fill knowledge gaps.
3. CONTRADICTION RESOLUTION: If different sources present conflicting numbers, dates, or accounts, outline both perspectives objectively. Do not pick a side unless one source is demonstrably more credible.
4. SYNTHESIS OVER SUMMARIZATION: Do not simply list summaries of search results. Synthesize the findings into a cohesive narrative structured with clear headers, logical groupings, and thematic continuity.

### TOOL-CALLING GUIDELINES
- Query Design: Keep search tool queries targeted and optimized. Avoid passing long conversational sentences to search engines. Use precise keywords, dates, or entity names.
- Deep Dive: If a search snippet provides a promising lead but lacks critical context, use the browser tool to read the full page before finalizing your synthesis.
- Temporal Context: Always keep the current year and context in mind to evaluate whether information is current or obsolete.

### OUTPUT STRUCTURE & STYLE
- Formality: Maintain a neutral, professional, and academic tone. Avoid conversational filler, personal pronouns, or fluff.
- Scannability: Use descriptive Markdown headers, bold key entities/metrics, and utilize bullet points for non-sequential data points.
- Citation: Always attach functional anchors or clear indicators mapping back to the specific source domain or URL retrieved by your tools."""

# Uncomment below to test the model with a simple prompt
"""
# Call and test the specified model
print("🤖 Agent analyzing query...")
response = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

response_message = response.choices[0].message
messages.append(response_message) # Append the model's response to keep history consistent

# Step 5: Check if the model requested a tool call
if response_message.tool_calls:
    for tool_call in response_message.tool_calls:
        if tool_call.function.name == "tavily_search":
            # Extract arguments generated by the model
            function_args = json.loads(tool_call.function.allocations or tool_call.function.arguments)
            search_query = function_args.get("query")
            
            print(f"🔍 Agent requested tool: tavily_search(query='{search_query}')")
            
            # Execute the actual web search using Tavily
            tavily_response = tavily_client.search(query=search_query, max_results=3)
            
            # Format results concisely for the LLM
            search_results_str = json.dumps(tavily_response.get("results", []))
            
            # Append the tool result to the conversation
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": "tavily_search",
                "content": search_results_str
            })
            
    # Step 6: Second LLM call - Send tool results back to get final answer
    print("🤖 Agent synthesizing final report...")
    final_response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )
    
    print("\n### Final Answer ###\n")
    print(final_response.choices[0].message.content)
else:
    # If the model didn't need tools, just print the direct answer
    print(response_message.content)
"""



'\n# Call and test the specified model\nresponse = client.chat.completions.create(\n    model=llm,     \n    messages=[\n        {"role": "system", "content": system_prompt},\n        {"role": "user", "content": "Give me a random fact about cats. Limit it to 100 tokens."}\n    ]\n)\n\n# Print the response text\nprint(response.choices[0].message.content)\n'